In [ ]:
from pathlib import Path

# ========== Paths ==========
DATA_DIR = Path("data")

FRAMES_DIR = DATA_DIR / "frames-1"     # data/frames-1/{video_id}/{000001.jpg...}
TRAIN_CSV  = DATA_DIR / "train_split.csv"
VALID_CSV  = DATA_DIR / "valid_split.csv"

OUT_DIR    = DATA_DIR / "embeddings" / "faces_emotiefflib_fps5_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = OUT_DIR / "index.parquet"

# ========== FPS ==========
ORIG_FPS   = 15
TARGET_FPS = 5
assert ORIG_FPS % TARGET_FPS == 0, "invalid stride"
FRAME_STRIDE = ORIG_FPS // TARGET_FPS

# ========== Face detection (MTCNN) ==========
DETECTOR_DEVICE = "cuda"
MTCNN_IMAGE_BATCH = 16
FACE_MIN_PROB = 0.90
MTCNN_MIN_FACE_SIZE = 40
MTCNN_THRESHOLDS = (0.6, 0.7, 0.7)
FACE_MARGIN = 0.20        # padding around bbox

# ========== EmotiEffLib encoder ==========
EMOTIEFF_ENGINE = "torch"
EMOTIEFF_MODEL  = "enet_b2_8"
ENCODER_DEVICE  = "cuda"

# ========== Encoding batching ==========
FACE_ENC_BATCH = 64
EMB_SAVE_DTYPE = "float16"

# ========== Output options ==========
SAVE_LOGITS = True
SKIP_IF_EXISTS = True

In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import cv2
import torch

from PIL import Image
from tqdm.auto import tqdm

from facenet_pytorch import MTCNN
from emotiefflib.facial_analysis import EmotiEffLibRecognizer, get_model_list

print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Face detector (MTCNN)
mtcnn = MTCNN(
    keep_all=True,
    device=DEVICE,
    thresholds=MTCNN_THRESHOLDS,
    min_face_size=MTCNN_MIN_FACE_SIZE,
    post_process=False,
)

# Encoder EmotiEffLib
assert EMOTIEFF_MODEL in set(get_model_list()), f"Unknown model {EMOTIEFF_MODEL}. Available: {get_model_list()}"
encoder = EmotiEffLibRecognizer(
    engine=EMOTIEFF_ENGINE,
    model_name=EMOTIEFF_MODEL,
    device=DEVICE if EMOTIEFF_ENGINE == "torch" else "cpu",
)

# Find out the embedding size D
dummy = np.zeros((224, 224, 3), dtype=np.uint8)
D = int(encoder.extract_features(dummy).shape[1])
print("Embedding dim D =", D, "| model =", EMOTIEFF_MODEL)

In [ ]:
from typing import List, Tuple, Optional, Dict, Any

def read_rgb(path: Path) -> np.ndarray:
    """Read image as RGB uint8 (H,W,3)."""
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Failed to read: {path}")
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return rgb

def parse_frame_number(path: Path) -> int:
    """000001.jpg -> 1"""
    return int(path.stem)

def list_frames(video_id: str) -> List[Path]:
    d = FRAMES_DIR / video_id
    if not d.exists():
        return []
    frames = sorted(d.glob("*.jpg"))
    return frames

def sample_frames_5fps(frames: List[Path]) -> List[Path]:
    """Assuming original extracted at ORIG_FPS, we take every FRAME_STRIDE frame."""
    if not frames:
        return []
    return frames[::FRAME_STRIDE]

def largest_box(boxes: np.ndarray) -> int:
    """Return index of largest bbox by area."""
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return int(np.argmax(areas))

def crop_face_with_margin(img_rgb: np.ndarray, box_xyxy: np.ndarray, margin: float) -> np.ndarray:
    """Crop face region with margin; output RGB uint8."""
    h, w = img_rgb.shape[:2]
    x1, y1, x2, y2 = box_xyxy.astype(np.float32)

    bw = x2 - x1
    bh = y2 - y1
    x1 = x1 - margin * bw
    y1 = y1 - margin * bh
    x2 = x2 + margin * bw
    y2 = y2 + margin * bh

    x1 = int(max(0, math.floor(x1)))
    y1 = int(max(0, math.floor(y1)))
    x2 = int(min(w, math.ceil(x2)))
    y2 = int(min(h, math.ceil(y2)))

    if x2 <= x1 or y2 <= y1:
        raise ValueError("Invalid crop box after margin")

    return img_rgb[y1:y2, x1:x2, :]

def chunked(seq, n: int):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]


In [ ]:
def process_one_video(video_id: str) -> Optional[Dict[str, Any]]:
    out_npz = OUT_DIR / f"{video_id}.npz"
    out_meta = OUT_DIR / f"{video_id}.json"

    if SKIP_IF_EXISTS and out_npz.exists():
        try:
            meta = json.loads(out_meta.read_text())
            meta["video_id"] = video_id
            meta["npz_path"] = str(out_npz)
            return meta
        except Exception:
            return {"video_id": video_id, "npz_path": str(out_npz), "skipped": True}

    frames = list_frames(video_id)
    if not frames:
        return None

    frames = sample_frames_5fps(frames)
    if not frames:
        return None

    T = len(frames)
    frame_numbers = np.array([parse_frame_number(p) for p in frames], dtype=np.int32)
    timestamps_sec = (frame_numbers - 1).astype(np.float32) / float(ORIG_FPS)

    # output buffers
    embeddings = np.zeros((T, D), dtype=np.float16)
    bbox_xyxy  = np.full((T, 4), -1.0, dtype=np.float32)
    face_prob  = np.zeros((T,), dtype=np.float32)
    face_found = np.zeros((T,), dtype=bool)

    # logits size K
    logits = None
    K = None

    # logits = features @ W^T+b, where W/b are stored inside encoder :contentReference[oaicite:5]{index=5}
    W = getattr(encoder, "classifier_weights", None)
    b = getattr(encoder, "classifier_bias", None)

    with torch.inference_mode():
        for batch_idx, batch_paths in enumerate(chunked(frames, MTCNN_IMAGE_BATCH)):
            batch_rgb = [read_rgb(p) for p in batch_paths]
            batch_pil = [Image.fromarray(im) for im in batch_rgb]

            boxes_list, probs_list = mtcnn.detect(batch_pil)

            face_imgs = []
            face_map  = []  # (global_frame_index, selected_box_xyxy, prob)

            for j, (img_rgb, boxes, probs) in enumerate(zip(batch_rgb, boxes_list, probs_list)):
                global_i = batch_idx * MTCNN_IMAGE_BATCH + j

                if boxes is None or probs is None:
                    continue

                # filtering by probability
                keep = probs >= FACE_MIN_PROB
                if not np.any(keep):
                    continue

                boxes_k = boxes[keep]
                probs_k = probs[keep]

                # we take the largest face
                k = largest_box(boxes_k)
                box = boxes_k[k]
                pr  = float(probs_k[k])

                try:
                    crop = crop_face_with_margin(img_rgb, box, FACE_MARGIN)
                except Exception:
                    continue

                face_imgs.append(crop)
                face_map.append((global_i, box.astype(np.float32), pr))

            # Encode found faces
            if face_imgs:
                feats_all = []
                for faces_chunk in chunked(face_imgs, FACE_ENC_BATCH):
                    feats = encoder.extract_features(faces_chunk)  # (n, D) float32 :contentReference[oaicite:6]{index=6}
                    feats_all.append(feats)
                feats_all = np.concatenate(feats_all, axis=0)

                # filling the buffers
                for (global_i, box, pr), feat in zip(face_map, feats_all):
                    face_found[global_i] = True
                    face_prob[global_i]  = pr
                    bbox_xyxy[global_i]  = box
                    embeddings[global_i] = feat.astype(np.float16)

                # logits
                if SAVE_LOGITS and (W is not None) and (b is not None):
                    # K it is determined from the weights of the classifier
                    if logits is None:
                        K = int(W.shape[0])
                        logits = np.zeros((T, K), dtype=np.float16)
                    # logits only for those shots where there is a face in this batch
                    f32 = feats_all.astype(np.float32)
                    l32 = (f32 @ W.T.astype(np.float32)) + b.astype(np.float32)
                    for (global_i, _, _), l in zip(face_map, l32):
                        logits[global_i] = l.astype(np.float16)

    meta = {
        "video_id": video_id,
        "npz_path": str(out_npz),
        "model": EMOTIEFF_MODEL,
        "engine": EMOTIEFF_ENGINE,
        "orig_fps": ORIG_FPS,
        "target_fps": TARGET_FPS,
        "frame_stride": FRAME_STRIDE,
        "embedding_dim": int(D),
        "frames_total": int(T),
        "faces_found": int(face_found.sum()),
        "save_logits": bool(SAVE_LOGITS and logits is not None),
        "logits_dim": int(K) if logits is not None else None,
    }

    # Save
    save_dict = dict(
        timestamps_sec=timestamps_sec,
        frame_numbers=frame_numbers,
        bbox_xyxy=bbox_xyxy,
        face_prob=face_prob,
        face_found=face_found,
        embeddings=embeddings,
    )
    if logits is not None:
        save_dict["emotion_logits"] = logits

    np.savez_compressed(out_npz, **save_dict)
    out_meta.write_text(json.dumps(meta, ensure_ascii=False, indent=2))

    return meta

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
valid_df = pd.read_csv(VALID_CSV)

all_ids = pd.concat([train_df[["Filename"]], valid_df[["Filename"]]], axis=0)["Filename"].astype(str).unique().tolist()
all_ids = sorted(all_ids)

print("Total videos:", len(all_ids))
print("Example ids:", all_ids[:5])

metas = []
failed = []

for vid in tqdm(all_ids, desc="Extract face embeddings"):
    try:
        m = process_one_video(vid)
        if m is None:
            failed.append({"video_id": vid, "reason": "no_frames"})
        else:
            metas.append(m)
    except Exception as e:
        failed.append({"video_id": vid, "reason": repr(e)})

meta_df = pd.DataFrame(metas)
fail_df = pd.DataFrame(failed)

display(meta_df.head())
print("OK:", len(meta_df), "Failed:", len(fail_df))

In [ ]:
meta_df.to_parquet(INDEX_PATH, index=False)
print("Saved index:", INDEX_PATH)

if len(fail_df):
    fail_path = OUT_DIR / "failed.csv"
    fail_df.to_csv(fail_path, index=False)
    print("Saved failures:", fail_path)

In [ ]:
# Test
test_id = meta_df.iloc[0]["video_id"]
npz_path = OUT_DIR / f"{test_id}.npz"

data = np.load(npz_path, allow_pickle=False)
print("keys:", data.files)
print("timestamps:", data["timestamps_sec"].shape, data["timestamps_sec"][:5])
print("embeddings:", data["embeddings"].shape, data["embeddings"].dtype)
print("faces found:", data["face_found"].sum(), "/", data["face_found"].shape[0])

emb = data["embeddings"].astype(np.float32)
mask = data["face_found"].astype(bool)
norms = np.linalg.norm(emb[mask], axis=1) if mask.any() else np.array([])
print("norms:", norms[:10], "mean:", norms.mean() if norms.size else None)